In [ ]:
#!/usr/bin/env python3
"""
AI Stock Analyzer  (A to Z)
---------------------------
Run:   python stock_analyzer.py
Then type a stock name, e.g.  RELIANCE   TCS   "tata motors"   AAPL

Does:
  1. Fundamental analysis  (valuation, profitability, growth, debt, cash flow, analysts)
  2. Technical analysis    (trend, RSI, MACD, Bollinger, ADX, ATR, volume, support/resistance)
  3. News sentiment        (simple keyword based)
  4. Final verdict         (STRONG BUY / BUY / HOLD / SELL / STRONG SELL)
  5. Trade plan            (buy zone, stop loss, 3 targets, trailing stop, position size)
  6. Exit rules            (when to sell)
  7. Optional Claude AI summary (if ANTHROPIC_API_KEY is set)
  8. Chart saved as PNG

Educational tool only. NOT financial advice. Always do your own research.
"""

import math
import os
import sys
import warnings

import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore")

# ======================= YOUR SETTINGS =======================
CAPITAL = 100000            # your total trading capital (in your currency)
RISK_PER_TRADE_PCT = 1.0    # risk max 1% of capital per trade
ATR_STOP_MULT = 2.0         # stop loss = entry - 2 x ATR
WEIGHT_FUNDAMENTAL = 0.40   # final score weight
WEIGHT_TECHNICAL = 0.60
SAVE_CHART = True
# =============================================================


# ----------------------- helpers -----------------------
def num(x):
    try:
        if x is None:
            return None
        v = float(x)
        return None if (math.isnan(v) or math.isinf(v)) else v
    except Exception:
        return None


def fmt(x, d=2, suffix=""):
    v = num(x)
    return "N/A" if v is None else f"{v:,.{d}f}{suffix}"


def pct(x, d=1):
    v = num(x)
    return "N/A" if v is None else f"{v * 100:.{d}f}%"


def big_money(x, currency):
    v = num(x)
    if v is None:
        return "N/A"
    if currency == "INR":
        return f"Rs {v / 1e7:,.0f} Cr"
    if v >= 1e12:
        return f"{v / 1e12:.2f} T {currency}"
    if v >= 1e9:
        return f"{v / 1e9:.2f} B {currency}"
    return f"{v / 1e6:.1f} M {currency}"


def line(char="=", n=78):
    print(char * n)


def title(text):
    print()
    line()
    print(f"  {text}")
    line()


def resolve_symbol(user_input):
    """Turn 'reliance' / 'tata motors' / 'AAPL' into a valid Yahoo symbol."""
    raw = user_input.strip()
    s = raw.upper().replace(" ", "")
    if "." in s or s.startswith("^"):
        candidates = [s]
    else:
        candidates = [s + ".NS", s + ".BO", s]  # NSE, BSE, then as-is (US etc.)
    for c in candidates:
        try:
            if not yf.Ticker(c).history(period="5d").empty:
                return c
        except Exception:
            pass
    # fall back to name search
    try:
        quotes = yf.Search(raw, max_results=8).quotes
        for q in quotes:
            sym = q.get("symbol", "")
            if sym.endswith(".NS") or sym.endswith(".BO"):
                return sym
        if quotes:
            return quotes[0]["symbol"]
    except Exception:
        pass
    return None


def score_block(rows):
    """rows = list of dicts with pts and mx. Returns 0-100 score."""
    if not rows:
        return 50.0
    total_pts = sum(r["pts"] for r in rows)
    total_max = sum(r["mx"] for r in rows)
    if total_max == 0:
        return 50.0
    return round((total_pts + total_max) / (2 * total_max) * 100, 1)


def print_rows(rows):
    for r in rows:
        sign = "+" if r["pts"] > 0 else ("-" if r["pts"] < 0 else "=")
        print(f"  [{sign}] {r['name']:<24} {r['value']:<16} {r['note']}")


# ----------------------- fundamentals -----------------------
def analyze_fundamentals(info, price):
    rows = []

    def add(name, value, pts, mx, note):
        rows.append({"name": name, "value": value, "pts": pts, "mx": mx, "note": note})

    sector = info.get("sector", "") or ""
    is_financial = "Financial" in sector

    # Valuation
    pe = num(info.get("trailingPE"))
    if pe is not None:
        if pe <= 0:
            add("P/E (trailing)", fmt(pe), -2, 2, "Company is loss making")
        elif pe < 15:
            add("P/E (trailing)", fmt(pe), 2, 2, "Cheap valuation")
        elif pe < 25:
            add("P/E (trailing)", fmt(pe), 1, 2, "Fair valuation")
        elif pe < 40:
            add("P/E (trailing)", fmt(pe), 0, 2, "Slightly expensive")
        else:
            add("P/E (trailing)", fmt(pe), -1, 2, "Expensive, needs high growth")

    fpe = num(info.get("forwardPE"))
    if fpe is not None and pe is not None and pe > 0 and fpe > 0:
        if fpe < pe * 0.9:
            add("Forward P/E", fmt(fpe), 1, 1, "Earnings expected to grow")
        elif fpe > pe * 1.1:
            add("Forward P/E", fmt(fpe), -1, 1, "Earnings expected to fall")

    peg = num(info.get("trailingPegRatio") or info.get("pegRatio"))
    if peg is not None and peg > 0:
        if peg < 1:
            add("PEG ratio", fmt(peg), 2, 2, "Growth is cheap")
        elif peg < 2:
            add("PEG ratio", fmt(peg), 0, 2, "Fairly priced for growth")
        else:
            add("PEG ratio", fmt(peg), -1, 2, "Pricey for growth")

    pb = num(info.get("priceToBook"))
    if pb is not None and pb > 0:
        limit_low, limit_high = (1.5, 3.5) if is_financial else (3, 7)
        if pb < limit_low:
            add("Price / Book", fmt(pb), 1, 1, "Trading near book value")
        elif pb > limit_high:
            add("Price / Book", fmt(pb), -1, 1, "High premium to book")

    # Profitability
    roe = num(info.get("returnOnEquity"))
    if roe is not None:
        if roe > 0.20:
            add("ROE", pct(roe), 2, 2, "Excellent")
        elif roe > 0.15:
            add("ROE", pct(roe), 1, 2, "Good")
        elif roe > 0.08:
            add("ROE", pct(roe), 0, 2, "Average")
        else:
            add("ROE", pct(roe), -2, 2, "Weak")

    roa = num(info.get("returnOnAssets"))
    if roa is not None and not is_financial:
        if roa > 0.10:
            add("ROA", pct(roa), 1, 1, "Efficient asset use")
        elif roa < 0.03:
            add("ROA", pct(roa), -1, 1, "Low asset efficiency")

    pm = num(info.get("profitMargins"))
    if pm is not None:
        if pm > 0.15:
            add("Net profit margin", pct(pm), 2, 2, "High margin business")
        elif pm > 0.08:
            add("Net profit margin", pct(pm), 1, 2, "Decent margin")
        elif pm > 0.03:
            add("Net profit margin", pct(pm), 0, 2, "Thin margin")
        else:
            add("Net profit margin", pct(pm), -1, 2, "Very thin / negative")

    om = num(info.get("operatingMargins"))
    if om is not None and om > 0.20:
        add("Operating margin", pct(om), 1, 1, "Strong operations")
    elif om is not None and om < 0.05:
        add("Operating margin", pct(om), -1, 1, "Weak operations")

    # Growth
    rg = num(info.get("revenueGrowth"))
    if rg is not None:
        if rg > 0.15:
            add("Revenue growth (YoY)", pct(rg), 2, 2, "Fast growth")
        elif rg > 0.08:
            add("Revenue growth (YoY)", pct(rg), 1, 2, "Healthy growth")
        elif rg >= 0:
            add("Revenue growth (YoY)", pct(rg), 0, 2, "Slow growth")
        else:
            add("Revenue growth (YoY)", pct(rg), -2, 2, "Shrinking sales")

    eg = num(info.get("earningsGrowth") or info.get("earningsQuarterlyGrowth"))
    if eg is not None:
        if eg > 0.15:
            add("Earnings growth", pct(eg), 2, 2, "Profits rising fast")
        elif eg > 0:
            add("Earnings growth", pct(eg), 1, 2, "Profits rising")
        else:
            add("Earnings growth", pct(eg), -2, 2, "Profits falling")

    # Financial health
    de = num(info.get("debtToEquity"))  # yfinance gives percent (45 = 0.45x)
    if de is not None and not is_financial:
        if de < 50:
            add("Debt / Equity", fmt(de / 100), 2, 2, "Low debt")
        elif de < 100:
            add("Debt / Equity", fmt(de / 100), 1, 2, "Manageable debt")
        elif de < 200:
            add("Debt / Equity", fmt(de / 100), -1, 2, "High debt")
        else:
            add("Debt / Equity", fmt(de / 100), -2, 2, "Very high debt")

    cr = num(info.get("currentRatio"))
    if cr is not None and not is_financial:
        if cr >= 1.5:
            add("Current ratio", fmt(cr), 1, 1, "Strong liquidity")
        elif cr < 1:
            add("Current ratio", fmt(cr), -1, 1, "Liquidity risk")

    fcf = num(info.get("freeCashflow"))
    if fcf is not None:
        if fcf > 0:
            add("Free cash flow", big_money(fcf, info.get("financialCurrency", "")), 1, 1, "Generates cash")
        else:
            add("Free cash flow", big_money(fcf, info.get("financialCurrency", "")), -1, 1, "Burning cash")

    # Dividend
    div_rate = num(info.get("dividendRate"))
    if div_rate and price:
        dy = div_rate / price
        if dy > 0.02:
            add("Dividend yield", pct(dy), 1, 1, "Pays good dividend")

    # Analysts
    target = num(info.get("targetMeanPrice"))
    if target and price:
        upside = (target - price) / price
        if upside > 0.15:
            add("Analyst target upside", pct(upside), 1, 1, f"Target {fmt(target)}")
        elif upside < -0.05:
            add("Analyst target upside", pct(upside), -1, 1, f"Target {fmt(target)}")
        else:
            add("Analyst target upside", pct(upside), 0, 1, f"Target {fmt(target)}")

    return rows, score_block(rows)


# ----------------------- technicals -----------------------
def compute_indicators(df):
    d = df.copy()
    c, h, l, v = d["Close"], d["High"], d["Low"], d["Volume"]

    for n in (20, 50, 200):
        d[f"SMA{n}"] = c.rolling(n).mean()
    d["EMA20"] = c.ewm(span=20, adjust=False).mean()

    # RSI (Wilder)
    delta = c.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_g = gain.ewm(alpha=1 / 14, adjust=False).mean()
    avg_l = loss.ewm(alpha=1 / 14, adjust=False).mean()
    rs = avg_g / avg_l.replace(0, np.nan)
    d["RSI"] = 100 - 100 / (1 + rs)

    # MACD
    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    d["MACD"] = ema12 - ema26
    d["MACD_SIG"] = d["MACD"].ewm(span=9, adjust=False).mean()
    d["MACD_HIST"] = d["MACD"] - d["MACD_SIG"]

    # ATR
    prev_c = c.shift()
    tr = pd.concat([h - l, (h - prev_c).abs(), (l - prev_c).abs()], axis=1).max(axis=1)
    d["ATR"] = tr.ewm(alpha=1 / 14, adjust=False).mean()

    # Bollinger
    std = c.rolling(20).std()
    d["BB_UP"] = d["SMA20"] + 2 * std
    d["BB_LO"] = d["SMA20"] - 2 * std

    # ADX
    up, down = h.diff(), -l.diff()
    plus_dm = pd.Series(np.where((up > down) & (up > 0), up, 0.0), index=d.index)
    minus_dm = pd.Series(np.where((down > up) & (down > 0), down, 0.0), index=d.index)
    plus_di = 100 * plus_dm.ewm(alpha=1 / 14, adjust=False).mean() / d["ATR"]
    minus_di = 100 * minus_dm.ewm(alpha=1 / 14, adjust=False).mean() / d["ATR"]
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)
    d["PLUS_DI"], d["MINUS_DI"] = plus_di, minus_di
    d["ADX"] = dx.ewm(alpha=1 / 14, adjust=False).mean()

    d["VOL20"] = v.rolling(20).mean()
    d["VOL5"] = v.rolling(5).mean()
    return d


def analyze_technicals(d):
    last, prev = d.iloc[-1], d.iloc[-2]
    price = last["Close"]
    rows = []

    def add(name, value, pts, mx, note):
        rows.append({"name": name, "value": value, "pts": pts, "mx": mx, "note": note})

    sma20, sma50, sma200 = num(last["SMA20"]), num(last["SMA50"]), num(last["SMA200"])

    if sma200 is not None:
        if price > sma200:
            add("Price vs 200 DMA", fmt(sma200), 2, 2, "Long-term uptrend")
        else:
            add("Price vs 200 DMA", fmt(sma200), -2, 2, "Long-term downtrend")
    if sma50 is not None:
        if price > sma50:
            add("Price vs 50 DMA", fmt(sma50), 1, 1, "Medium-term bullish")
        else:
            add("Price vs 50 DMA", fmt(sma50), -1, 1, "Medium-term bearish")
    if sma50 is not None and sma200 is not None:
        if sma50 > sma200:
            add("50 DMA vs 200 DMA", "Golden", 1, 1, "50 DMA above 200 DMA")
        else:
            add("50 DMA vs 200 DMA", "Death", -1, 1, "50 DMA below 200 DMA")
    if sma20 is not None:
        if price > sma20:
            add("Price vs 20 DMA", fmt(sma20), 1, 1, "Short-term bullish")
        else:
            add("Price vs 20 DMA", fmt(sma20), -1, 1, "Short-term bearish")

    rsi = num(last["RSI"])
    if rsi is not None:
        if rsi < 30:
            add("RSI (14)", fmt(rsi, 1), 2, 2, "Oversold, bounce likely")
        elif rsi < 45:
            add("RSI (14)", fmt(rsi, 1), 0, 2, "Weak momentum")
        elif rsi <= 65:
            add("RSI (14)", fmt(rsi, 1), 1, 2, "Healthy momentum")
        elif rsi <= 75:
            add("RSI (14)", fmt(rsi, 1), 0, 2, "Strong but getting hot")
        else:
            add("RSI (14)", fmt(rsi, 1), -2, 2, "Overbought, pullback risk")

    macd, sig = num(last["MACD"]), num(last["MACD_SIG"])
    if macd is not None and sig is not None:
        if macd > sig and prev["MACD"] <= prev["MACD_SIG"]:
            add("MACD", "Bull cross", 2, 2, "Fresh bullish crossover")
        elif macd > sig:
            add("MACD", "Bullish", 1, 2, "MACD above signal")
        elif macd < sig and prev["MACD"] >= prev["MACD_SIG"]:
            add("MACD", "Bear cross", -2, 2, "Fresh bearish crossover")
        else:
            add("MACD", "Bearish", -1, 2, "MACD below signal")

    bb_up, bb_lo = num(last["BB_UP"]), num(last["BB_LO"])
    if bb_up is not None and bb_lo is not None:
        if price < bb_lo:
            add("Bollinger Bands", "Below lower", 1, 1, "Stretched down, mean reversion")
        elif price > bb_up:
            add("Bollinger Bands", "Above upper", -1, 1, "Stretched up, may cool off")

    adx = num(last["ADX"])
    if adx is not None:
        bullish_di = last["PLUS_DI"] > last["MINUS_DI"]
        if adx > 25:
            add("ADX trend strength", fmt(adx, 1), 1 if bullish_di else -1, 1,
                "Strong uptrend" if bullish_di else "Strong downtrend")
        else:
            add("ADX trend strength", fmt(adx, 1), 0, 1, "Sideways / weak trend")

    v5, v20 = num(last["VOL5"]), num(last["VOL20"])
    if v5 and v20:
        up_move = price > d["Close"].iloc[-6]
        if v5 > v20 * 1.2 and up_move:
            add("Volume", f"{v5 / v20:.2f}x avg", 1, 1, "Rising price on rising volume")
        elif v5 > v20 * 1.2 and not up_move:
            add("Volume", f"{v5 / v20:.2f}x avg", -1, 1, "Falling price on rising volume")

    support20 = float(d["Low"].tail(20).min())
    support60 = float(d["Low"].tail(60).min())
    resistance20 = float(d["High"].tail(20).max())
    resistance60 = float(d["High"].tail(60).max())
    hi52 = float(d["High"].tail(252).max())
    lo52 = float(d["Low"].tail(252).min())

    levels = {
        "support20": support20, "support60": support60,
        "resistance20": resistance20, "resistance60": resistance60,
        "hi52": hi52, "lo52": lo52,
    }
    return rows, score_block(rows), levels


# ----------------------- news sentiment -----------------------
POSITIVE = ["surge", "jump", "rally", "gain", "profit", "beat", "record", "growth", "upgrade", "buy",
            "strong", "wins", "order", "expansion", "dividend", "bullish", "outperform", "soars", "rises"]
NEGATIVE = ["fall", "drop", "plunge", "loss", "miss", "decline", "downgrade", "sell", "weak", "probe",
            "fraud", "penalty", "lawsuit", "cut", "bearish", "underperform", "slump", "crash", "debt", "fine"]


def news_sentiment(ticker):
    headlines = []
    try:
        for item in (ticker.news or [])[:12]:
            t = item.get("title") or (item.get("content") or {}).get("title")
            if t:
                headlines.append(t)
    except Exception:
        pass
    if not headlines:
        return [], 0, "No news found"
    score = 0
    for h in headlines:
        low = h.lower()
        score += sum(w in low for w in POSITIVE) - sum(w in low for w in NEGATIVE)
    label = "Positive" if score > 1 else ("Negative" if score < -1 else "Neutral")
    return headlines, score, label


# ----------------------- quarterly trend -----------------------
def quarterly_trend(ticker):
    try:
        q = ticker.quarterly_income_stmt
        if q is None or q.empty:
            return None
        out = []
        for col in list(q.columns)[:4]:
            rev = num(q.loc["Total Revenue", col]) if "Total Revenue" in q.index else None
            ni = num(q.loc["Net Income", col]) if "Net Income" in q.index else None
            out.append((str(col.date()), rev, ni))
        return out
    except Exception:
        return None


# ----------------------- trade plan -----------------------
def build_trade_plan(d, levels, verdict, fund_score, tech_score):
    last = d.iloc[-1]
    price = float(last["Close"])
    atr = float(last["ATR"])
    ema20 = float(last["EMA20"])
    rsi = num(last["RSI"]) or 50

    # Entry logic
    overheated = rsi > 72 or price > ema20 + 2.5 * atr
    if overheated:
        entry = ema20
        entry_note = "Price is stretched. Wait for a pullback to the 20 EMA zone."
    else:
        entry = price
        entry_note = "Price is acceptable. Can enter near current market price."

    buy_zone = (ema20 - 0.5 * atr, ema20 + 0.5 * atr)
    breakout = levels["resistance60"] * 1.002

    # Stop loss: ATR stop, or just below swing support if that is tighter but sensible
    sl_atr = entry - ATR_STOP_MULT * atr
    sl_support = levels["support20"] - 0.25 * atr
    stop = sl_atr
    if sl_atr < sl_support < entry - 1.0 * atr:
        stop = sl_support

    risk = entry - stop
    t1, t2, t3 = entry + 1.5 * risk, entry + 2.5 * risk, entry + 4.0 * risk

    risk_amount = CAPITAL * RISK_PER_TRADE_PCT / 100
    qty = int(risk_amount // risk) if risk > 0 else 0
    max_qty_by_capital = int(CAPITAL // entry) if entry > 0 else 0
    qty = max(0, min(qty, max_qty_by_capital))

    trailing = float(d["Close"].tail(22).max()) - 3 * atr  # chandelier exit

    return {
        "price": price, "entry": entry, "entry_note": entry_note,
        "buy_zone": buy_zone, "breakout": breakout,
        "stop": stop, "risk": risk, "risk_pct": risk / entry * 100,
        "t1": t1, "t2": t2, "t3": t3,
        "qty": qty, "capital_used": qty * entry, "risk_amount": risk_amount,
        "trailing": trailing, "atr": atr,
    }


def decide(final_score, tech_score, rsi):
    if final_score >= 70:
        v = "STRONG BUY"
    elif final_score >= 58:
        v = "BUY"
    elif final_score >= 42:
        v = "HOLD / WAIT"
    elif final_score >= 30:
        v = "SELL / AVOID"
    else:
        v = "STRONG SELL"
    # safety overrides
    if v in ("STRONG BUY", "BUY") and tech_score < 40:
        v = "HOLD / WAIT"
    if v in ("STRONG BUY", "BUY") and rsi and rsi > 75:
        v = "WAIT FOR PULLBACK (overbought)"
    return v


# ----------------------- chart -----------------------
def save_chart(d, symbol, plan, verdict):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        x = d.tail(180)
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True,
                                       gridspec_kw={"height_ratios": [3, 1]})
        ax1.plot(x.index, x["Close"], label="Close", color="black", linewidth=1.4)
        ax1.plot(x.index, x["SMA50"], label="SMA50", color="blue", linewidth=1)
        ax1.plot(x.index, x["SMA200"], label="SMA200", color="red", linewidth=1)
        ax1.fill_between(x.index, x["BB_LO"], x["BB_UP"], color="grey", alpha=0.12)
        ax1.axhline(plan["entry"], color="green", linestyle="--", label=f"Entry {plan['entry']:.2f}")
        ax1.axhline(plan["stop"], color="red", linestyle="--", label=f"Stop {plan['stop']:.2f}")
        for i, t in enumerate((plan["t1"], plan["t2"], plan["t3"]), 1):
            ax1.axhline(t, color="darkgreen", linestyle=":", label=f"T{i} {t:.2f}")
        ax1.set_title(f"{symbol}  |  {verdict}")
        ax1.legend(loc="upper left", fontsize=8, ncol=3)
        ax2.plot(x.index, x["RSI"], color="purple")
        ax2.axhline(70, color="red", linestyle=":")
        ax2.axhline(30, color="green", linestyle=":")
        ax2.set_ylabel("RSI")
        plt.tight_layout()
        fname = f"{symbol.replace('.', '_')}_analysis.png"
        plt.savefig(fname, dpi=110)
        plt.close()
        return fname
    except Exception as e:
        return None


# ----------------------- optional Claude AI summary -----------------------
def ai_summary(report_text):
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        return None
    try:
        import anthropic
        client = anthropic.Anthropic(api_key=key)
        msg = client.messages.create(
            model="claude-sonnet-5-5",
            max_tokens=900,
            messages=[{
                "role": "user",
                "content": (
                    "You are a careful equity analyst. Below is an automated analysis of a stock. "
                    "Write a short plain-English summary: 1) business quality, 2) key risks, "
                    "3) whether the trade plan looks sensible, 4) what would change the view. "
                    "Be balanced, no guarantees.\n\n" + report_text
                ),
            }],
        )
        return "".join(b.text for b in msg.content if getattr(b, "type", "") == "text")
    except Exception as e:
        return f"(AI summary skipped: {e})"


# ----------------------- main analysis -----------------------
def analyze(user_input):
    symbol = resolve_symbol(user_input)
    if not symbol:
        print(f"Could not find a stock for '{user_input}'. Try the exact ticker (e.g. RELIANCE, TCS, AAPL).")
        return

    t = yf.Ticker(symbol)
    hist = t.history(period="2y", interval="1d", auto_adjust=True)
    if hist is None or len(hist) < 60:
        print("Not enough price history to analyze this stock.")
        return
    try:
        info = t.info or {}
    except Exception:
        info = {}

    d = compute_indicators(hist)
    price = float(d["Close"].iloc[-1])
    currency = info.get("currency", "")
    report = []  # collect key text for AI summary

    # ---------- Company overview ----------
    title(f"COMPANY OVERVIEW  -  {info.get('longName', symbol)} ({symbol})")
    print(f"  Sector / Industry : {info.get('sector', 'N/A')} / {info.get('industry', 'N/A')}")
    print(f"  Current price     : {fmt(price)} {currency}")
    print(f"  Market cap        : {big_money(info.get('marketCap'), currency)}")
    print(f"  52w High / Low    : {fmt(info.get('fiftyTwoWeekHigh'))} / {fmt(info.get('fiftyTwoWeekLow'))}")
    print(f"  Beta (volatility) : {fmt(info.get('beta'))}")
    print(f"  Insider/Promoter  : {pct(info.get('heldPercentInsiders'))}   "
          f"Institutions: {pct(info.get('heldPercentInstitutions'))}")
    print(f"  Analyst view      : {info.get('recommendationKey', 'N/A')} "
          f"({info.get('numberOfAnalystOpinions', 'N/A')} analysts)")
    summary = info.get("longBusinessSummary")
    if summary:
        print(f"\n  About: {summary[:450]}...")
    report.append(f"Stock: {info.get('longName', symbol)}, sector {info.get('sector')}, price {price:.2f} {currency}")

    # ---------- Fundamentals ----------
    f_rows, f_score = analyze_fundamentals(info, price)
    title(f"FUNDAMENTAL ANALYSIS   Score: {f_score}/100")
    if f_rows:
        print_rows(f_rows)
    else:
        print("  Fundamental data not available for this stock.")
    qt = quarterly_trend(t)
    if qt:
        print("\n  Last quarters (latest first):")
        for dt, rev, ni in qt:
            print(f"    {dt}   Revenue: {big_money(rev, info.get('financialCurrency', currency))}"
                  f"   Net profit: {big_money(ni, info.get('financialCurrency', currency))}")
    report.append("Fundamentals: " + "; ".join(f"{r['name']}={r['value']} ({r['note']})" for r in f_rows))

    # ---------- Technicals ----------
    t_rows, t_score, levels = analyze_technicals(d)
    title(f"TECHNICAL ANALYSIS   Score: {t_score}/100")
    print_rows(t_rows)
    print(f"\n  Support (20d / 60d)    : {fmt(levels['support20'])} / {fmt(levels['support60'])}")
    print(f"  Resistance (20d / 60d) : {fmt(levels['resistance20'])} / {fmt(levels['resistance60'])}")
    print(f"  52w High / Low         : {fmt(levels['hi52'])} / {fmt(levels['lo52'])}"
          f"   (price is {(price / levels['hi52'] - 1) * 100:.1f}% from high)")
    report.append("Technicals: " + "; ".join(f"{r['name']}={r['value']} ({r['note']})" for r in t_rows))

    # ---------- News ----------
    headlines, n_score, n_label = news_sentiment(t)
    title(f"NEWS SENTIMENT   {n_label} (raw {n_score:+d})")
    for h in headlines[:8]:
        print(f"  - {h}")
    if not headlines:
        print("  No recent headlines available.")

    # ---------- Final verdict ----------
    final = WEIGHT_FUNDAMENTAL * f_score + WEIGHT_TECHNICAL * t_score
    if n_label == "Positive":
        final += 2
    elif n_label == "Negative":
        final -= 2
    final = round(max(0, min(100, final)), 1)
    rsi = num(d["RSI"].iloc[-1])
    verdict = decide(final, t_score, rsi)
    plan = build_trade_plan(d, levels, verdict, f_score, t_score)

    title("FINAL RESULT")
    print(f"  Fundamental score : {f_score}/100   (weight {int(WEIGHT_FUNDAMENTAL * 100)}%)")
    print(f"  Technical score   : {t_score}/100   (weight {int(WEIGHT_TECHNICAL * 100)}%)")
    print(f"  News sentiment    : {n_label}")
    print(f"  COMBINED SCORE    : {final}/100")
    print(f"\n  >>>  VERDICT: {verdict}  <<<")

    # ---------- Trade plan ----------
    title("TRADE PLAN  (swing trade, days to weeks)")
    if verdict.startswith("SELL") or verdict.startswith("STRONG SELL"):
        print("  Do NOT buy now. If you already hold, consider exiting or tightening your stop.")
        print(f"  Re-check when price closes above 50 DMA ({fmt(d['SMA50'].iloc[-1])}) with rising volume.")
    else:
        print(f"  When to BUY:")
        print(f"    1) {plan['entry_note']}")
        print(f"    2) Pullback buy zone : {fmt(plan['buy_zone'][0])} to {fmt(plan['buy_zone'][1])}")
        print(f"    3) Breakout buy      : close above {fmt(plan['breakout'])} with volume above average")
        print(f"\n  Entry price      : {fmt(plan['entry'])}")
        print(f"  STOP LOSS        : {fmt(plan['stop'])}   (risk {plan['risk_pct']:.1f}% per share, "
              f"{ATR_STOP_MULT} x ATR={fmt(plan['atr'])})")
        print(f"  Target 1 (1:1.5) : {fmt(plan['t1'])}   -> sell 40%, move stop to entry")
        print(f"  Target 2 (1:2.5) : {fmt(plan['t2'])}   -> sell 30%, trail the rest")
        print(f"  Target 3 (1:4)   : {fmt(plan['t3'])}   -> sell remaining or trail")
        print(f"\n  Position sizing  : capital {CAPITAL:,.0f}, risk {RISK_PER_TRADE_PCT}% = {plan['risk_amount']:,.0f}")
        print(f"  Quantity to buy  : {plan['qty']} shares  (invest about {plan['capital_used']:,.0f})")
        print(f"  Trailing stop now: {fmt(plan['trailing'])}  (highest close 22d - 3 x ATR, only move UP)")

    print("\n  When to SELL (exit rules):")
    print(f"    - Price closes below stop loss {fmt(plan['stop'])}  -> exit full position, no excuses")
    print(f"    - Price hits targets T1/T2/T3 -> book profit in parts as above")
    print(f"    - Daily close below trailing stop -> exit")
    print(f"    - Close below 50 DMA ({fmt(d['SMA50'].iloc[-1])}) after a big rally -> exit swing trade")
    print(f"    - RSI above 75 and then turns down, or fresh MACD bearish crossover -> book profits")
    print(f"    - Bad news: fraud, regulatory action, big profit miss, promoter selling -> reduce/exit")

    print("\n  Key risk notes:")
    print(f"    - Never risk more than {RISK_PER_TRADE_PCT}% of capital on one trade.")
    print("    - Check results/earnings dates before entering; gaps can jump over stop loss.")
    print("    - Market direction matters: avoid new buys when Nifty/S&P is below its 200 DMA.")

    # ---------- chart ----------
    if SAVE_CHART:
        fname = save_chart(d, symbol, plan, verdict)
        if fname:
            print(f"\n  Chart saved: {fname}")

    # ---------- optional AI ----------
    report.append(f"Scores: fundamental {f_score}, technical {t_score}, news {n_label}, combined {final}, verdict {verdict}")
    report.append(f"Plan: entry {plan['entry']:.2f}, stop {plan['stop']:.2f}, targets {plan['t1']:.2f}/{plan['t2']:.2f}/{plan['t3']:.2f}")
    ai = ai_summary("\n".join(report))
    if ai:
        title("AI SUMMARY (Claude)")
        print(ai)

    print()
    line("-")
    print("  Educational tool only. Not financial advice. Markets are risky, you can lose money.")
    line("-")


def main():
    # Ignore hidden arguments that Colab/Jupyter add (e.g. "-f kernel.json")
    args = [a for a in sys.argv[1:] if not a.startswith("-") and not a.endswith(".json")]
    in_notebook = "ipykernel" in sys.modules
    if args and not in_notebook:
        analyze(" ".join(args))
        return
    print("AI Stock Analyzer  (type 'q' to quit)")
    while True:
        name = input("\nEnter stock name / symbol: ").strip()
        if name.lower() in ("q", "quit", "exit", ""):
            break
        try:
            analyze(name)
        except Exception as e:
            print(f"Error while analyzing: {e}")


if __name__ == "__main__":
    main()